In [1]:
import pandas as pd
import numpy as np
import glob
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, average_precision_score

In [2]:
all_files = glob.glob("../data/PRICE_AND_DEMAND_*_SA1.csv")
print(f"Found {len(all_files)} files")

df = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)
df["SETTLEMENTDATE"] = pd.to_datetime(df["SETTLEMENTDATE"])
df = df[df["REGION"] == "SA1"].sort_values("SETTLEMENTDATE").reset_index(drop=True)
print(df.shape)
print(df["SETTLEMENTDATE"].dt.to_period("M").value_counts().sort_index())

Found 12 files
(103680, 5)
SETTLEMENTDATE
2023-01    8927
2023-02    8064
2023-03       1
2023-12    8927
2024-01    8928
2024-02    8352
2024-03       1
2024-12    8927
2025-01    8928
2025-02    8064
2025-03       1
2025-12    8927
2026-01    8928
2026-02    8064
2026-03       1
2026-06    8639
2026-07       1
Freq: M, Name: count, dtype: int64


In [3]:
df["hour"] = df["SETTLEMENTDATE"].dt.hour
df["day_of_year"] = df["SETTLEMENTDATE"].dt.dayofyear
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["doy_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365)
df["doy_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365)

In [4]:
INTERVALS_PER_HOUR = 12
THRESHOLD = 200
LOOKAHEAD = 3 * INTERVALS_PER_HOUR

df["will_breach"] = (
    df["TOTALDEMAND"].shift(-1).rolling(window=LOOKAHEAD, min_periods=LOOKAHEAD).min() < THRESHOLD
).astype(int)

df["demand_lag_30m"] = df["TOTALDEMAND"].shift(6)
df["demand_lag_1h"] = df["TOTALDEMAND"].shift(12)
df["demand_lag_2h"] = df["TOTALDEMAND"].shift(24)
df["demand_trend"] = df["TOTALDEMAND"] - df["demand_lag_2h"]

gap_minutes = df["SETTLEMENTDATE"].diff().dt.total_seconds() / 60
is_gap_nearby = (
    gap_minutes.rolling(LOOKAHEAD, min_periods=1).max().gt(5)
    | gap_minutes.shift(-LOOKAHEAD).rolling(LOOKAHEAD, min_periods=1).max().gt(5)
)
df = df[~is_gap_nearby.fillna(True)]
df = df.dropna().reset_index(drop=True)
print(df.shape)
print(f"Positive class rate: {df['will_breach'].mean():.3%}")

(103368, 16)
Positive class rate: 4.707%


In [5]:
df["year"] = df["SETTLEMENTDATE"].dt.year
print(df.groupby("year")["will_breach"].agg(["sum", "count", "mean"]))

       sum  count      mean
year                       
2023   833  25823  0.032258
2024  1405  26136  0.053757
2025  1730  25848  0.066930
2026   898  25561  0.035132


In [6]:
train = df[df["year"] <= 2025]
test = df[df["year"] == 2026]
print(f"Train: {len(train)} rows, breach rate {train['will_breach'].mean():.3%}")
print(f"Test: {len(test)} rows, breach rate {test['will_breach'].mean():.3%}")

Train: 77807 rows, breach rate 5.100%
Test: 25561 rows, breach rate 3.513%


In [7]:
features = ["hour_sin", "hour_cos", "doy_sin", "doy_cos",
            "demand_lag_30m", "demand_lag_1h", "demand_lag_2h", "demand_trend"]

X_train, y_train = train[features], train["will_breach"]
X_test, y_test = test[features], test["will_breach"]

model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"Average precision (PR-AUC): {average_precision_score(y_test, y_proba):.3f}")

              precision    recall  f1-score   support

           0       1.00      0.99      0.99     24663
           1       0.72      0.94      0.81       898

    accuracy                           0.98     25561
   macro avg       0.86      0.96      0.90     25561
weighted avg       0.99      0.98      0.99     25561

Average precision (PR-AUC): 0.934
